## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?  
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.


**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Justin Diamond

**Date:** 9/30/2026

**Q1.**
1. What is the difference between regression and classification?

The difference between regression and classification is that regression is predicting a numerical value and classification is sorting a piece of data into a category. In regression there are infinitely many possible values it can output to be, while classification is putting it into a fixed set of possible categories.  

2. What is a confusion table? What does it help us understand about a model's performance?

A confusion table compares a model’s predicted classifications with the actual classifications. This is used to help see how many of the predictions are correct and what kind of errors the model is making.

3. What does the SSE quantify about a particular model?

The Sum of Squared Errors quantifies how far a model’s predictions are from the actual values. It does this by squaring each prediction error and adding them together. The lower the SSE value, the more accurate it is because that means the model has smaller errors. 


4. What are overfitting and underfitting? 

When training a model, it is overfitting when it learns the training data way too closely, including noise and random patterns. This makes it work well on the training data but poorly on new data. Underfitting happens when a model is too basic to understand the real patterns in the data. It performs poorly on both training and new data. In KNN, a very small k more likely to overfit because predictions depend heavily on individual nearby points while a large k is more likely to underfit because there are too many neighbors. 

5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?

Splitting the data into training and testing sets and choosing a good value of $k$ based on accuracy or SSE on the test set improves the model performance because it allows us to evaluate how well the model can process to data it has not already seen. By comparing different values of $k$, we can find a good middle ground to avoid overfitting and underfitting and performs best on new data.

6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

Reporting a single class label is easy to interpret because it gives one prediction output, but it does not show how confident the model is in that prediction. Reporting a probability distribution over class labels gives us more information by showing how likely each class is, which can be useful when the prediction is not fully sure. The downside is that probability distributions are more complex to interpret and depend on the model being accurate on probabilities.



**Q2**

Part 1

In [23]:
import pandas as pd

df = pd.read_csv("data/land_mines.csv")
print(df.head())
df.info()
print(df["mine_type"].value_counts().sort_index())
df.describe()

    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1
<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


,voltage,height,soil,mine_type
count,338.000000,338.000000,338.000000,338.000000
mean,0.430634,0.508876,0.503550,2.952663
std,0.195819,0.306043,0.344244,1.419703
min,0.197734,0.000000,0.000000,1.000000
25%,0.309737,0.272727,0.200000,2.000000
50%,0.359516,0.545455,0.600000,3.000000
75%,0.482628,0.727273,0.800000,4.000000
max,0.999999,1.000000,1.000000,5.000000


The data has 338 mines and no missing values. The label mine_type takes values 1 through 5, with 71, 70, 66, 66, and 65 mines of each type. This is good as the classes are roughly balanced. The features are voltage, height, and soil. All three are already on a 0 to 1 scale. Voltage is mostly from 0.2 to 0.5, with a mean near 0.43. Height and soil only take a few fixed values: soil is 0, 0.2, 0.4, 0.6, 0.8, or 1, and height is one of 12 evenly spaced values from 0 to 1. 

Part 2

In [24]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts().sort_index())
print(y_test.value_counts().sort_index())

(169, 3) (169, 3)
mine_type
1    35
2    35
3    33
4    33
5    33
Name: count, dtype: int64
mine_type
1    36
2    35
3    33
4    33
5    32
Name: count, dtype: int64


Part 3 $k$-NN classifier

In [25]:
from sklearn.neighbors import KNeighborsClassifier

for k in range(1, 171, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    print(k, knn.score(X_test, y_test))

1 0.40236686390532544
3 0.378698224852071
5 0.3136094674556213
7 0.3136094674556213
9 0.34911242603550297
11 0.35502958579881655
13 0.34911242603550297
15 0.33136094674556216
17 0.3609467455621302
19 0.33727810650887574
21 0.3609467455621302
23 0.3254437869822485
25 0.35502958579881655
27 0.3668639053254438
29 0.3727810650887574
31 0.38461538461538464
33 0.3609467455621302
35 0.3609467455621302
37 0.3668639053254438
39 0.378698224852071
41 0.40236686390532544
43 0.38461538461538464
45 0.3727810650887574
47 0.35502958579881655
49 0.3254437869822485
51 0.33136094674556216
53 0.3609467455621302
55 0.3431952662721893
57 0.41420118343195267
59 0.38461538461538464
61 0.3609467455621302
63 0.3431952662721893
65 0.33727810650887574
67 0.3431952662721893
69 0.3727810650887574
71 0.3668639053254438
73 0.3727810650887574
75 0.378698224852071
77 0.3668639053254438
79 0.35502958579881655
81 0.3609467455621302
83 0.378698224852071
85 0.378698224852071
87 0.39644970414201186
89 0.40236686390532544
91

We can choose $k=57$ for the model, as we tried every odd k from 1 through 169 and chose 57 because it had the highest test accuracy, about $41.4\%$.

Part 4 

In [26]:
from sklearn.metrics import confusion_matrix, accuracy_score

knn = KNeighborsClassifier(n_neighbors=57)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.41420118343195267
[[21  0  3  7  5]
 [ 1 24  4  4  2]
 [ 8  1 10 12  2]
 [13  0  7  9  4]
 [14  0  4  8  6]]


The model is about 41.4% accurate on the test set which is 70 of 169 mines. That accuracy is uneven, as type 2 is right 24 of 35 times, and type 1 is right 21 of 36 times. However, type 3 is right only 10 of 33 times, type 4 is 9 of 33 times, and even worse type 5 is 6 of 32 times. Overall, type 3 is often called type 4 or type 1, and types 4 and 5 are often called type 1. Predictions that outputted type 2 were were only wrong once. 

Part 5

I would advise someone to be cautious of what they use this model for. The one I'd trust enough to use is a mine of type 2, since when it predicted type 2 the 25 times, it was correct 24 times. Type 1 has it opposite problem. It correctly predcits 21 real type 1 mines, but it predicts type 1 about 57 times, and most of those mines are really type 3, 4, or 5. Overall, the only mine type that was correctly labeled enough to be safely used was type 2 mines. 

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 045b15e97e66c739f3dc8d6fadc3d55060405ab8
- **AI tool and model used:** GPT 5.6 Sol

### *Prompts used

1. Review my answers to Phase 1 of this assignment. Help me improve upon my answers by proposing a list of concise changes to make. 

### *AI-proposed changes


1. Clarify that selecting $k$ on the test set biases the reported accuracy; use cross-validation or a validation set, then evaluate once on an untouched test set.
2. Limit the $k$ search to a reasonable range and report the selection method clearly.
3. Note that SSE is mainly for regression and is only comparable across the same dataset/sample size.
4. Add per-class precision and recall to support the confusion-matrix discussion.
5. Remove the claim that type 2 predictions are “safe”; the sample is small, recall is only $24/35$, and land-mine decisions require human verification.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Modify | I call the holdout data a validation set because it was used to select k. Therefore, 41.4% is validation accuracy, not unbiased test accuracy. |
| 2 | Reject | Trying every valid odd k is systematic, and the assignment does not require a restricted range. |
| 3 | Accept | SSE is mainly used for regression and depends on sample size, so comparisons should use the same data or MSE. |
| 4 | Accept | Precision and recall provide class-specific information that overall accuracy does not show. I will calculate them using classification_report. |
| 5 | Accept | Calling type 2 predictions safe is unsupported. Its recall is only 24/35, and predictions should not replace expert verification. |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

**Q1.**
1. What is the difference between regression and classification?

The difference between regression and classification is that regression is predicting a numerical value and classification is sorting a piece of data into a category. In regression there are infinitely many possible values it can output to be, while classification is putting it into a fixed set of possible categories.  

2. What is a confusion table? What does it help us understand about a model's performance?

A confusion table compares a model’s predicted classifications with the actual classifications. This is used to help see how many of the predictions are correct and what kind of errors the model is making.

3. What does the SSE quantify about a particular model?

The Sum of Squared Errors (SSE), mainly used for regression, measures the total squared difference between predicted and actual values. A lower SSE means smaller prediction errors on the same dataset. Because SSE increases with the number of observations, it should only be compared using datasets of the same size, or a normalized measure such as MSE should be used.


4. What are overfitting and underfitting? 

When training a model, it is overfitting when it learns the training data way too closely, including noise and random patterns. This makes it work well on the training data but poorly on new data. Underfitting happens when a model is too basic to understand the real patterns in the data. It performs poorly on both training and new data. In KNN, a very small k more likely to overfit because predictions depend heavily on individual nearby points while a large k is more likely to underfit because there are too many neighbors. 

5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?

Splitting the data into training and testing sets and choosing a good value of $k$ based on accuracy or SSE on the test set improves the model performance because it allows us to evaluate how well the model can perform on data it has not already seen. By comparing different values of $k$, we can find a good middle ground to avoid overfitting and underfitting and perform the best on new data. Its score is validation accuracy rather than an unbiased estimate of final test performance. Ideally, an untouched test set would be used for the final evaluation.

6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

Reporting a single class label is easy to interpret because it gives one prediction output, but it does not show how confident the model is in that prediction. Reporting a probability distribution over class labels gives us more information by showing how likely each class is, which can be useful when the prediction is not fully sure. The downside is that probability distributions are more complex to interpret and depend on the model being accurate on probabilities.



**Q2**

Part 1

In [85]:
import pandas as pd

df = pd.read_csv("data/land_mines.csv")
print(df.head())
df.info()
print(df["mine_type"].value_counts().sort_index())
df.describe()

    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1
<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64


,voltage,height,soil,mine_type
count,338.000000,338.000000,338.000000,338.000000
mean,0.430634,0.508876,0.503550,2.952663
std,0.195819,0.306043,0.344244,1.419703
min,0.197734,0.000000,0.000000,1.000000
25%,0.309737,0.272727,0.200000,2.000000
50%,0.359516,0.545455,0.600000,3.000000
75%,0.482628,0.727273,0.800000,4.000000
max,0.999999,1.000000,1.000000,5.000000


The data has 338 mines and no missing values. The label mine_type takes values 1 through 5, with 71, 70, 66, 66, and 65 mines of each type. This is good as the classes are roughly balanced. The features are voltage, height, and soil. All three are already on a 0 to 1 scale. Voltage is mostly from 0.2 to 0.5, with a mean near 0.43. Height and soil only take a few fixed values: soil is 0, 0.2, 0.4, 0.6, 0.8, or 1, and height is one of 12 evenly spaced values from 0 to 1. 

Part 2

In [86]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts().sort_index())
print(y_test.value_counts().sort_index())

(169, 3) (169, 3)
mine_type
1    35
2    35
3    33
4    33
5    33
Name: count, dtype: int64
mine_type
1    36
2    35
3    33
4    33
5    32
Name: count, dtype: int64


Part 3 $k$-NN classifier

In [87]:
from sklearn.neighbors import KNeighborsClassifier

for k in range(1, 171, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    print(k, knn.score(X_test, y_test))

1 0.40236686390532544
3 0.378698224852071
5 0.3136094674556213
7 0.3136094674556213
9 0.34911242603550297
11 0.35502958579881655
13 0.34911242603550297
15 0.33136094674556216
17 0.3609467455621302
19 0.33727810650887574
21 0.3609467455621302
23 0.3254437869822485
25 0.35502958579881655
27 0.3668639053254438
29 0.3727810650887574
31 0.38461538461538464
33 0.3609467455621302
35 0.3609467455621302
37 0.3668639053254438
39 0.378698224852071
41 0.40236686390532544
43 0.38461538461538464
45 0.3727810650887574
47 0.35502958579881655
49 0.3254437869822485
51 0.33136094674556216
53 0.3609467455621302
55 0.3431952662721893
57 0.41420118343195267
59 0.38461538461538464
61 0.3609467455621302
63 0.3431952662721893
65 0.33727810650887574
67 0.3431952662721893
69 0.3727810650887574
71 0.3668639053254438
73 0.3727810650887574
75 0.378698224852071
77 0.3668639053254438
79 0.35502958579881655
81 0.3609467455621302
83 0.378698224852071
85 0.378698224852071
87 0.39644970414201186
89 0.40236686390532544
91

We can choose $k=57$ for the model, as we tried every odd k from 1 through 169 and chose 57 because it had the highest validation accuracy, about $41.4\%$.

Part 4 

In [88]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

knn = KNeighborsClassifier(n_neighbors=57)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


0.41420118343195267
[[21  0  3  7  5]
 [ 1 24  4  4  2]
 [ 8  1 10 12  2]
 [13  0  7  9  4]
 [14  0  4  8  6]]
              precision    recall  f1-score   support

           1       0.37      0.58      0.45        36
           2       0.96      0.69      0.80        35
           3       0.36      0.30      0.33        33
           4       0.23      0.27      0.25        33
           5       0.32      0.19      0.24        32

    accuracy                           0.41       169
   macro avg       0.45      0.41      0.41       169
weighted avg       0.45      0.41      0.42       169



The model is about 41.4% accurate on the validation set which is 70 of 169 mines. That accuracy is uneven, as type 2 is right 24 of 35 times, and type 1 is right 21 of 36 times. However, type 3 is right only 10 of 33 times, type 4 is 9 of 33 times, and even worse type 5 is 6 of 32 times. Overall, type 3 is often called type 4 or type 1, and types 4 and 5 are often called type 1. Predictions that outputted type 2 were only wrong once. The classification report confirms that type 2 has the highest precision (0.96) and recall (0.69), while type 4 has the lowest precision (0.23) and type 5 has the lowest recall (0.19).


Part 5

I would advise someone to be cautious of what they use this model for. The one I'd cautiously trust the prediction is a mine of type 2, since when it predicted type 2 the 25 times, it was correct 24 times. However, while type 2 predictions appear most reliable, they should still be treated cautiously. Type 1 has the opposite problem. It correctly predicts 21 real type 1 mines, but it predicts type 1 about 57 times, and most of those mines are really type 3, 4, or 5. Overall, the only mine type that was correctly labeled enough to be safely used was type 2 mines. 

### *Reflection on AI assistance
In your own words without generative AI.

The AI helped improve my solution by making the explanation of SSE more precise, helping explain the difference of validation accuracy from unbiased test accuracy, and adding precision and recall for each mine type. The classification report made the model’s uneven performance clearer than its overall accuracy. I also changed my recommendation wording because it told me how describing type 2 predictions as “safe” was not justified in this case. Although type 2 had 0.96 precision, its recall was only 0.69, meaning the model missed 11 of 35 actual type 2 mines.

One of AI's limitations was the recommendation to restrict the range of k values. I rejected this because testing every valid odd value was a valid system, and the assignment did not require a smaller range. I verified the accepted changes by rerunning the notebook and comparing the classification report with the confusion matrix, and they aligned. 

My only remaining concern is that the model had only a 41.4% validation accuracy and performs poorly for several mine types. Therefore, this model should only support other actual assessment. 


